# P3 - Fig Engine real-path memory verification

Measures process RSS by phase for `FigModel.from_pretrained()` and Tier-1 training. The default is `lowram`, the mode relevant to the minimum-memory claim. Partial results are written atomically to Google Drive after every phase.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
P3_DIR = '/content/drive/MyDrive/littlefig-p3'
os.makedirs(P3_DIR, exist_ok=True)
print('Durable P3 directory:', P3_DIR)

In [ ]:
import os, subprocess
REPO = '/content/littlefig'
BRANCH = 'research/p1-figmezo-verify'
if not os.path.exists(REPO + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, 'https://github.com/Harboria-Labs/littlefig.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'switch', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '-q', '--ff-only'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', REPO], check=True)
os.chdir(REPO)
print('Source checkout ready:', BRANCH)

In [ ]:
MEMORY_MODE = 'lowram'  # lowram tests the minimum-memory claim; also try figcache or fast
STEPS = 20
BATCH_SIZE = 2
SEQUENCE_LENGTH = 256
RESULTS_PATH = P3_DIR + '/figengine_8gb_' + MEMORY_MODE + '_results.json'
cmd = [
    'python', '-u', 'benchmark/experiment_8gb_v1.py',
    '--memory-mode', MEMORY_MODE, '--steps', str(STEPS),
    '--batch-size', str(BATCH_SIZE), '--sequence-length', str(SEQUENCE_LENGTH),
    '--memory-budget-gib', '8.0', '--claim-memory-gib', '0.4',
    '--results-path', RESULTS_PATH,
]
print('Run:', ' '.join(cmd))

In [ ]:
# Stream every child-process line to Colab and Drive. This remains useful if
# the benchmark exits nonzero or the runtime kills it for memory pressure.
import subprocess, os, json, time
LOG_PATH = P3_DIR + '/figengine_8gb_' + MEMORY_MODE + '.log'
print('Starting benchmark...', flush=True)
print('Durable log:', LOG_PATH, flush=True)
print('Command:', ' '.join(cmd), flush=True)
with open(LOG_PATH, 'w', buffering=1) as log_file:
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
        log_file.write(line)
    process.stdout.close()
    RETURN_CODE = process.wait()
print('Benchmark return code:', RETURN_CODE, flush=True)
print('Partial/final results exist:', os.path.exists(RESULTS_PATH), flush=True)
if os.path.exists(RESULTS_PATH):
    print(json.dumps(json.load(open(RESULTS_PATH)), indent=2), flush=True)
if RETURN_CODE != 0:
    print('Benchmark failed. The output above and durable log contain the cause.', flush=True)

In [ ]:
# Post-run live-data summary and visualizations.
import json, os, re
import matplotlib.pyplot as plt
print('Results:', RESULTS_PATH, os.path.exists(RESULTS_PATH))
if os.path.exists(RESULTS_PATH):
    result = json.load(open(RESULTS_PATH))
    print(json.dumps(result, indent=2))
    phases = result.get('phase_stats', {})
    phase_names = list(phases)
    phase_rss = [phases[p].get('rss_peak_gib', 0) for p in phase_names]
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes[0, 0].bar(phase_names, phase_rss, color='#3b82f6')
    axes[0, 0].axhline(result.get('budget_gib', 8.0), color='red', linestyle='--', label='RAM budget')
    axes[0, 0].set_title('Peak RSS by phase'); axes[0, 0].set_ylabel('GiB'); axes[0, 0].tick_params(axis='x', rotation=35); axes[0, 0].legend()
    steps, losses, lrs, speeds, rss = [], [], [], [], []
    if os.path.exists(LOG_PATH):
        for line in open(LOG_PATH, errors='replace'):
            m = re.search(r'step=\s*(\d+)\s+loss=([0-9.eE+-]+)\s+lr=([0-9.eE+-]+)\s+speed=([0-9.eE+-]+)', line)
            if m:
                steps.append(int(m.group(1))); losses.append(float(m.group(2))); lrs.append(float(m.group(3))); speeds.append(float(m.group(4)))
            m = re.search(r'FigQuant \[(\d+)/(\d+)\].*rss=([0-9.]+)GiB', line)
            if m: rss.append((int(m.group(1)), float(m.group(3))))
    if steps:
        axes[0, 1].plot(steps, losses, marker='o'); axes[0, 1].set_title('Training loss'); axes[0, 1].set_xlabel('Step'); axes[0, 1].set_ylabel('Loss')
        axes[1, 0].plot(steps, speeds, marker='o', color='#16a34a'); axes[1, 0].set_title('Training speed'); axes[1, 0].set_xlabel('Step'); axes[1, 0].set_ylabel('Steps/s')
        axes[1, 1].plot(steps, lrs, marker='o', color='#f97316'); axes[1, 1].set_title('Learning rate'); axes[1, 1].set_xlabel('Step'); axes[1, 1].set_ylabel('LR')
    else:
        axes[0, 1].text(0.5, 0.5, 'No training-step telemetry found', ha='center', va='center'); axes[1, 0].axis('off'); axes[1, 1].axis('off')
    plt.tight_layout()
    GRAPH_PATH = P3_DIR + '/figengine_8gb_' + MEMORY_MODE + '_telemetry.png'
    plt.savefig(GRAPH_PATH, dpi=160, bbox_inches='tight')
    print('Saved telemetry graph:', GRAPH_PATH)
    plt.show()
    if rss:
        print('Quantization RSS samples: first=', rss[0], 'last=', rss[-1], 'peak=', max(rss, key=lambda x: x[1]))
else:
    print('No result JSON yet; durable log:', LOG_PATH)